# 08 - Track B: finding-purpose triage

**Purpose:** Test whether a small model can suggest operation_purpose for scanner findings better than the rule engine; chart model vs baselines, per-fold accuracy and class balance.

**Inputs:** data/labels/public_code (team annotations, single annotator) + fetched public repos in data/public/raw

**Outputs:** metric chart, per-fold chart, class-balance chart

**Status labels:** VERIFIED run; small sample (row and repo counts printed below). Suggestion only; never overrides rules, HOLD or ML-KEM/ML-DSA roles.

Context: the demo bank in `demo_bank_app/` is SYNTHETIC. Charts are also saved to `scratch/derived/member3/figs/`.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys, json, subprocess
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "cryptoshift").is_dir())
for p in (str(ROOT / "src"), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
DERIVED = ROOT / "scratch" / "derived" / "member3"
FIGS = DERIVED / "figs"
FIGS.mkdir(parents=True, exist_ok=True)
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
PY = sys.executable

def ensure(rel_path, script, *args):
    """Run a project script once if its output file is missing (keeps notebooks runnable in a fresh checkout)."""
    target = ROOT / rel_path
    if not target.is_file():
        print(f"{rel_path} missing -> running {script} ...")
        subprocess.run([PY, str(ROOT / "scripts" / script), *args], cwd=ROOT, check=True)
    return target

def savefig(name):
    plt.tight_layout(); plt.savefig(FIGS / f"{name}.png", dpi=120); plt.show()
# This notebook reads no credentials. If one is ever needed, read it from an environment variable at runtime and
# never display it.

In [ ]:
raw = ROOT / 'data' / 'public' / 'raw'
if not any(p.is_dir() for p in raw.iterdir()):
    subprocess.run([PY, str(ROOT / 'scripts' / 'fetch_public_label_sources.py')], cwd=ROOT, check=True)
from cryptoshift.ml import purpose_triage
r = purpose_triage.run()
print(r['rows'], 'rows |', r['repos'])
print(r['decision'])

In [ ]:
met = pd.DataFrame(r['metrics_pooled_over_folds']).T[['accuracy','macro_f1']].rename(index={'model':'decision tree','majority_baseline':'majority','rule_engine':'rule engine (kept)'})
folds = pd.DataFrame(r['folds']).set_index('held_out_repo'); folds['accuracy'] = folds['model_correct'] / folds['n_test']
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
met.plot(kind='bar', ax=ax[0], color=['#457b9d','#e9c46a'], title='Leave-one-repository-out (pooled)'); ax[0].set_ylim(0, 1); ax[0].tick_params(axis='x', rotation=0)
for cont in ax[0].containers: ax[0].bar_label(cont, fmt='%.2f', fontsize=8)
folds['accuracy'].plot(kind='bar', ax=ax[1], color='#2a9d8f', title='Model accuracy per held-out repo'); ax[1].set_ylim(0, 1)
for i, (n, v) in enumerate(zip(folds['n_test'], folds['accuracy'])): ax[1].text(i, v, f'n={n}', ha='center', va='bottom', fontsize=8)
pd.Series(r['class_balance']).sort_values().plot(kind='barh', ax=ax[2], color='#8d99ae', title=f"Label purpose balance ({r['rows']} rows)")
savefig('08_purpose_triage')
met

In [ ]:
pd.Series(r['on_rule_unknown_findings']).to_frame('findings where the rule engine says unknown')